# 1.1 章节介绍：昇腾开发基础

欢迎来到 **《从0到1构建FA》** 课程的第一章！

在第2章里，我们将彻底搞懂 FlashAttention 的数学原理；在第3篇里，我们将亲手在昇腾 NPU 上写出一个 FA 算子。但在那之前，必须先回答一个最朴素的问题：

> **我们到底是在什么硬件上写代码？这块硬件“听得懂”哪些指令？**

就像赛车手要先熟悉赛道、厨师要先用好灶台一样——不懂数据在芯片里怎么流，写出的算子就只能“能跑”；懂数据流，才能“跑快”。

本章就是这张**赛道地图**：昇腾 950 的架构与芯片参数，再到 Ascend C Kernel 的六大基础积木（数据流、内存、搬运、Cube、Vector、同步）。

## 课前 3 分钟：这门课为什么存在

大模型的骨架是 Transformer，而 Transformer 的心脏是**注意力机制**：

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

一句话解读：每个位置的 Query 去“查询”所有位置的 Key，按匹配度对 Value 加权求和。它让序列中任意两个位置都能**一步直达**地建立依赖——RNN 要逐步传递、CNN 受感受野限制，Attention 没有这些束缚。

代价则写在复杂度里：计算量随序列长度**平方增长**，且推理时每个头都要缓存历史的 K、V 向量（**KV Cache**）。

KV Cache 有多重？LLM 推理时，模型参数的显存占用是个常数，而 KV Cache 随上下文长度**线性增长**——长上下文场景下它会反超参数、吃掉大部分显存。于是工业界一路演进：

| 机制 | KV 共享方式 | 效果 |
|--|--|--|
| **MHA**（多头注意力） | 每个头独立一份 KV | 表达力最强，KV Cache 最大 |
| **GQA**（分组查询） | 一组头共享一份 KV | 折中：显存大降、精度小损 |
| **MQA**（多查询） | 所有头共享一份 KV | 显存最省，精度损失略增 |

另一条普遍规律是**计算密度**（每读 1 字节数据能支撑多少次有效计算）：密度低则**带宽受限**——算力空转，瓶颈在搬数；密度高则**计算受限**——瓶颈在算力本身。标准 Attention 恰是前者的典型：softmax(QK^T)V 反复读写显存里的大矩阵，大量时间花在搬运上。

FlashAttention 的出发点就一句话：**分块计算（Tiling）+ 在线归一化（Online Softmax），让中间结果尽量留在片上高速存储，极大减少显存读写**——把带宽受限的算子推向计算受限。

而“留在片上”——留到哪一层、谁来搬、谁来算、怎么同步——全是硬件架构问题。于是本课程主线成立：

- **第2章**：讲清 FA 的数学（分块递推为什么成立）；
- **第1章**（本章）：讲清硬件（数据在 AI Core 里怎么旅行）；
- **第3篇起**：把两者拼成真正可运行的算子，再一路优化到底。

## 本章学习目标

学完第1章，你将能够：

1. **说出达芬奇架构的核心设计思想**：为什么“3D Cube 计算单元”是 AI 芯片的关键武器；
2. **背出 950 的“装备清单”**：AI Core 规模、存储容量、带宽、算力等关键参数的量级；
3. **画出算子的三条流**：指令流、同步信号流、计算数据流——任何 Ascend C Kernel 都是这三条流的编织物；
4. **讲清数据的一次完整旅行**：GM → L1 → L0A/L0B → L0C → UB → GM，以及每一段由哪条搬运指令（MTE2/MTE1/MTE3/FIXPIPE）完成；
5. **建立 FA 的硬件直觉**：QK^T 与 P·V 是 Cube 的活，Online Softmax 是 Vector 的活——这个“两次 Cube 夹一次 Vector”的结构贯穿整门课；
6. **说清 SPMD 多核模型**：一份代码启动 N 个 Block，block_idx 决定数据归属——FA 多核切分的第一层。

## 前置知识

| 知识点 | 要求 | 说明 |
|--|--|--|
| C/C++ | 会读基本语法 | 第2部分起会读 Kernel 代码，本章只看伪代码 |
| 计算机体系结构 | 知道“缓存比内存快” | 有缓存/内存层级的概念即可 |
| 线性代数 | 知道矩阵乘法 | 理解 Cube 为什么为矩阵而生 |
| NPU / CANN | **不要求** | 本章从零讲起 |

> **本章全程无需 NPU 环境**：以原理讲解与示意图为主，任意电脑都可以阅读。若想“眼见为实”，可在有昇腾硬件的环境（如 CANNLab）中运行各节的 `npu-smi info` 观察真实设备。

## 章节内容导航

本章共 9 个 Notebook，分两块：**昇腾架构基础**（1.2）与 **Ascend C 开发基础**（2.1 ~ 2.6）：

| Notebook | 主题 | 核心内容 |
|--|--|--|
| [1.1 章节介绍](01.01_chapter_intro.ipynb)（本页） | 导学 | 目标、前置、路线图 |
| [1.2 昇腾 950 架构](01.02_ascend_950_arch.ipynb) | 装备清单 | 架构演进（耦合→分离）、三代规格对比、SIMD/SIMT、CV 融合、GM 访问通路、FA 硬件优化 |
| [2.1 算子数据流](01.03_operator_dataflow.ipynb) | 积木一 | SPMD 多核模型、三条流、搬运-计算-搬运、FA ↔ 硬件映射 |
| [2.2 内存结构](01.04_memory_structure.ipynb) | 积木二 | 存储层级、私有领地、ND/分形排布 |
| [2.3 搬运指令](01.05_data_move_instructions.ipynb) | 积木三 | MTE2/MTE1/MTE3/FIXPIPE、NDDMA、双缓冲 |
| [2.4 Cube 计算指令](01.06_cube_instructions.ipynb) | 积木四 | 脉动阵列、mmad、M/N/K |
| [2.5 Vector 计算指令](01.07_vector_instructions.ipynb) | 积木五 | Repeat/Mask/Stride、API 分级、FA 的 softmax 指令 |
| [2.6 同步指令](01.08_sync_instructions.ipynb) | 积木六 | SetFlag/WaitFlag、TQue 队列 |
| [1.9 章节测试](01.09_chapter_test.ipynb) | 检验 | 全章综合测试 |

### 学习主线：从芯片到积木，从积木到 FA

```text
1.2  先认识芯片（950 为 LLM 重构）
        │
        ▼
2.1 ~ 2.6  再拆开 AI Core，拿到六块积木：
            数据怎么流（2.1）· 放在哪里（2.2）· 谁来搬运（2.3）
            谁算矩阵（2.4）· 谁算向量（2.5）· 谁来协同（2.6）
        │
        ▼
第2章之后    用这六块积木，拼出 FlashAttention
            QK^T（Cube）→ Online Softmax（Vector）→ P·V（Cube）
```

> 每一节都会把“当前积木”与 FA 挂钩——这不是跑题，而是让你在第3篇写 Kernel 时，能立刻想起“这段计算该交给谁”。

In [ ]:
# 环境自检：本章为纯阅读章节，只需 Python 环境即可
import sys
print('Python 版本:', sys.version.split()[0])
print('环境就绪，开始 1.2 节的学习吧！')

## 学习建议

- **图优先**：本章大量使用架构图，先看图建立空间感，再读文字补细节；
- **两清单一地图**：1.2/1.3 的参数清单与 2.1 的数据流图，建议学完后能默画出来；
- **每节练一练**：课后练习 3 题用于自检，答案在 `answer/` 目录，先做后看；
- **不必死记参数**：参数是量级直觉（“UB 是 KB 级、GM 是 GB 级”），不是背诵题。

